In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

ROOT = Path.cwd().parent
RESULTS_DIR = ROOT / "results"
MODELS_DIR = ROOT / "models"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Project:", ROOT)
print("Device:", device)
print("Results folder:", RESULTS_DIR.exists())
print("Models folder:", MODELS_DIR.exists())


Project: c:\Users\HIMANSHU\Desktop\minor project 1\Minor-Project
Device: cuda
Results folder: True
Models folder: True


In [2]:

face_metadata_df = pd.read_csv(
    RESULTS_DIR / "final_face_metadata.csv"
)

face_metadata_df = face_metadata_df[
    face_metadata_df["file_exists"].astype(str).str.lower() == "true"
].copy()

train_df = face_metadata_df[
    face_metadata_df["split"] == "train"
].reset_index(drop=True)

val_df = face_metadata_df[
    face_metadata_df["split"] == "val"
].reset_index(drop=True)

test_df = face_metadata_df[
    face_metadata_df["split"] == "test"
].reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Unique test videos:", test_df["video_id"].nunique())


Train: 39450
Validation: 9870
Test: 12321
Unique test videos: 1233


In [3]:

IMG_SIZE = 224

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class VideoFaceDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = Path(row["face_path"])

        if not image_path.is_absolute():
            image_path = ROOT / image_path

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(
            float(row["label"]),
            dtype=torch.float32
        )

        return (
            image,
            label,
            str(row["video_path"]),
            str(row["video_id"])
        )

test_dataset = VideoFaceDataset(test_df, test_transform)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Test images:", len(test_dataset))
print("Test batches:", len(test_loader))


Test images: 12321
Test batches: 771


In [4]:

checkpoint_path = MODELS_DIR / "week4_resnet50.pth"

resnet50 = models.resnet50(weights=None)
resnet50.fc = nn.Linear(resnet50.fc.in_features, 1)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

if "state_dict" in checkpoint:
    checkpoint = checkpoint["state_dict"]
elif "model_state_dict" in checkpoint:
    checkpoint = checkpoint["model_state_dict"]

checkpoint = {
    k.replace("module.", ""): v
    for k, v in checkpoint.items()
}

resnet50.load_state_dict(checkpoint)
resnet50 = resnet50.to(device)
resnet50.eval()

print("ResNet50 loaded successfully!")


ResNet50 loaded successfully!


In [5]:

face_predictions = []

with torch.no_grad():
    for batch_no, (images, labels, video_paths, video_ids) in enumerate(test_loader):

        images = images.to(device)

        outputs = resnet50(images)
        probabilities = torch.sigmoid(outputs).squeeze(1)

        for i in range(len(labels)):
            face_predictions.append({
                "video_path": video_paths[i],
                "video_id": video_ids[i],
                "true_label": int(labels[i].item()),
                "probability_fake": float(probabilities[i].item())
            })

        if (batch_no + 1) % 100 == 0:
            print(f"Processed {batch_no + 1}/{len(test_loader)} batches")

face_predictions_df = pd.DataFrame(face_predictions)

print("\nFace-level predictions:", len(face_predictions_df))
print("Unique videos:", face_predictions_df["video_path"].nunique())


Processed 100/771 batches
Processed 200/771 batches
Processed 300/771 batches
Processed 400/771 batches
Processed 500/771 batches
Processed 600/771 batches
Processed 700/771 batches

Face-level predictions: 12321
Unique videos: 1233


In [6]:

video_predictions_df = (
    face_predictions_df
    .groupby(["video_path", "video_id"], as_index=False)
    .agg(
        true_label=("true_label", "first"),
        probability_fake=("probability_fake", "mean"),
        face_count=("probability_fake", "count")
    )
)

video_predictions_df["predicted_label"] = (
    video_predictions_df["probability_fake"] >= 0.14
).astype(int)

print("Total videos:", len(video_predictions_df))
print("\nFace counts per video:")
print(video_predictions_df["face_count"].describe())

print("\nVideo-level predictions:")
print(video_predictions_df.head())


Total videos: 1233

Face counts per video:
count    1233.000000
mean        9.992701
std         0.256307
min         1.000000
25%        10.000000
50%        10.000000
75%        10.000000
max        10.000000
Name: face_count, dtype: float64

Video-level predictions:
                                          video_path   video_id  true_label  \
0  c:\Users\Raja Kumar\OneDrive\Desktop\DEEP FAKE...   id0_0001           0   
1  c:\Users\Raja Kumar\OneDrive\Desktop\DEEP FAKE...   id0_0004           0   
2  c:\Users\Raja Kumar\OneDrive\Desktop\DEEP FAKE...  id10_0003           0   
3  c:\Users\Raja Kumar\OneDrive\Desktop\DEEP FAKE...  id10_0004           0   
4  c:\Users\Raja Kumar\OneDrive\Desktop\DEEP FAKE...  id11_0001           0   

   probability_fake  face_count  predicted_label  
0          0.483757          10                1  
1          0.552127          10                1  
2          0.055924          10                0  
3          0.000007          10                0  


In [7]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

y_true = video_predictions_df["true_label"]
y_pred = video_predictions_df["predicted_label"]
y_prob = video_predictions_df["probability_fake"]

print("VIDEO-LEVEL RESULTS")
print("=" * 40)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall:", recall_score(y_true, y_pred, zero_division=0))
print("F1-Score:", f1_score(y_true, y_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_true, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Real", "Fake"],
    zero_division=0
))


VIDEO-LEVEL RESULTS
Accuracy: 0.9091646390916464
Precision: 0.9122502172024327
Recall: 0.9896324222431668
F1-Score: 0.9493670886075949
ROC-AUC: 0.8493824386822437

Confusion Matrix:
[[  71  101]
 [  11 1050]]

Classification Report:
              precision    recall  f1-score   support

        Real       0.87      0.41      0.56       172
        Fake       0.91      0.99      0.95      1061

    accuracy                           0.91      1233
   macro avg       0.89      0.70      0.75      1233
weighted avg       0.91      0.91      0.89      1233



In [8]:

video_output_path = RESULTS_DIR / "week9_video_level_predictions.csv"

video_predictions_df.to_csv(
    video_output_path,
    index=False
)

print("Video-level predictions saved!")
print(video_output_path)


Video-level predictions saved!
c:\Users\HIMANSHU\Desktop\minor project 1\Minor-Project\results\week9_video_level_predictions.csv


In [9]:

video_metrics = pd.DataFrame([{
    "Evaluation": "Video-Level",
    "Threshold": 0.14,
    "Total_Videos": len(video_predictions_df),
    "Accuracy": accuracy_score(y_true, y_pred),
    "Precision": precision_score(y_true, y_pred, zero_division=0),
    "Recall": recall_score(y_true, y_pred, zero_division=0),
    "F1_Score": f1_score(y_true, y_pred, zero_division=0),
    "ROC_AUC": roc_auc_score(y_true, y_prob)
}])

metrics_path = RESULTS_DIR / "week9_video_level_metrics.csv"
video_metrics.to_csv(metrics_path, index=False)

print(video_metrics.to_string(index=False))
print("\nSaved:", metrics_path)


 Evaluation  Threshold  Total_Videos  Accuracy  Precision   Recall  F1_Score  ROC_AUC
Video-Level       0.14          1233  0.909165    0.91225 0.989632  0.949367 0.849382

Saved: c:\Users\HIMANSHU\Desktop\minor project 1\Minor-Project\results\week9_video_level_metrics.csv


In [10]:

face_metrics = pd.DataFrame([{
    "Evaluation": "Face-Level",
    "Accuracy": 0.906826,
    "Precision": 0.913766,
    "Recall": 0.984731,
    "F1-Score": 0.947922,
    "ROC-AUC": 0.833104
}])

video_metrics_comparison = pd.DataFrame([{
    "Evaluation": "Video-Level",
    "Accuracy": accuracy_score(y_true, y_pred),
    "Precision": precision_score(y_true, y_pred, zero_division=0),
    "Recall": recall_score(y_true, y_pred, zero_division=0),
    "F1-Score": f1_score(y_true, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_true, y_prob)
}])

comparison_df = pd.concat(
    [face_metrics, video_metrics_comparison],
    ignore_index=True
)

comparison_df.to_csv(
    RESULTS_DIR / "week9_face_vs_video_comparison.csv",
    index=False
)

print(comparison_df.to_string(index=False))
print("\nComparison saved successfully!")


 Evaluation  Accuracy  Precision   Recall  F1-Score  ROC-AUC
 Face-Level  0.906826   0.913766 0.984731  0.947922 0.833104
Video-Level  0.909165   0.912250 0.989632  0.949367 0.849382

Comparison saved successfully!
